In [ ]:
%load_ext autoreload
%autoreload 2

# Incorporation of GA and NN (reflectance only)

In [ ]:
import os
import sys
import pandas as pd
from dotenv import load_dotenv
#from Code.Arnold_models import PAR_ModellingFunctions_pipelines
from Code.Utils.util_methods import UtilMethods
from keras.models import load_model
from Code.NN.utils.keras_functions import binary_focal_loss
import pickle
import shutil
from tqdm.notebook import tqdm
from Code.GA.GA_scripts import run_ga
from datetime import datetime
from zoneinfo import ZoneInfo
from Code.GANN.utils.utils import summarize_results_GANN

#sys.modules['PAR_ModellingFunctions_pipelines'] = PAR_ModellingFunctions_pipelines

base = UtilMethods.find_project_root(os.getcwd())
print(f"Project root found: {base}")

if load_dotenv(f'{base}/.env'):
    print(".env found")
else:
    print("ERROR .env not found")

## Variables

In [ ]:
NN_MODEL_DIR = 'prc-split-val-small-threshold-04/tuned_model_435'
NN_MODEL_PATH = f'{base}/Code/NN/Results/Tuning/{NN_MODEL_DIR}'
THRESHOLD = 0.4
RESULT_FOLDER = f'{base}/Code/GANN/Results/{NN_MODEL_DIR}'
CONFIDENCE_OCCURENCE = True # used the confidence of the predictions instead of the occurences
STRICT_PIGMENT_USE = True # only uses the predicted pigments (confidence above threshold)

## Import the data

In [ ]:
X = pd.read_csv(f'{base}/Dataset/traintest/X.csv')
y = pd.read_csv(f'{base}/Dataset/traintest/y.csv')
all_pigments = list(y.columns)

## Load NN model

In [ ]:
NN_model = load_model(f"{NN_MODEL_PATH}/model.keras", custom_objects={"loss": binary_focal_loss(gamma=2.0, alpha=0.8)})

## Load Lab prediction model

In [ ]:
with open(f'{base}/Dataset/Arnold/GP_pipeline_models_dict_16April2025.pkl', 'rb') as f:
    pipeline_dict_file = pickle.load(f)


LAB_model =  pipeline_dict_file['model']['curve']

## Run the GA with pigment prediction

In [ ]:

NN_result = NN_model.predict(X, verbose=1)
NN_result_bin = (NN_result >= THRESHOLD).astype(int)
NN_result_bin_df = pd.DataFrame(NN_result_bin, columns=all_pigments)
NN_result_df = pd.DataFrame(NN_result, columns=all_pigments)

step = 10

for i in tqdm(range(0, len(X)), desc=f'Iterating through all recipes by steps of {step}'):

    save_main_dir = f'{RESULT_FOLDER}/recipe_{i}'

    if os.path.exists(save_main_dir):
        if os.path.exists(f'{save_main_dir}/.done'):
            continue
        else:
            # delte the folder and recreate later
            shutil.rmtree(save_main_dir)

    if i % step != 0:
        continue

    row = y.iloc[[i]]

    count_per_row = (row > 0.0).sum(axis=1).iloc[0]
    if count_per_row <= 1:
        continue
    
    row_df = NN_result_bin_df.loc[[i]]
    columns_to_use = list(row_df.loc[:, row_df.iloc[0] != 0].columns)

    possibilities = [True, False]

    os.makedirs(save_main_dir, exist_ok=True)

    # save the model predictions
    NN_result_df.iloc[[i]].to_csv(f'{save_main_dir}/model_prediction.csv', index=False)
    NN_result_df[columns_to_use].iloc[[i]].to_csv(f'{save_main_dir}/predicted_pigments.csv', index=False)

    c=0
    additions = ['sp-co','sp','co','-']
    for a in possibilities:
        for b in possibilities:
            
            strict_pigment_use = a
            confidence_occurence = b

            save_folder = f'{save_main_dir}/{additions[c]}'

            if strict_pigment_use:
                max_pigment_values_row = max_pigment_values[columns_to_use]
                if confidence_occurence:
                    occurences_row = NN_result_df.loc[i, columns_to_use]
                else:
                    occurences_row = occurences[columns_to_use]
            else:
                max_pigment_values_row = max_pigment_values
                if confidence_occurence:
                    occurences_row = NN_result_df.iloc[i]
                else:
                    continue
        

            for j in range(3):
                run_ga(X.iloc[[i]], LAB_model, max_pigment_values_row, occurences_row, pipeline_dict_file, all_pigments, population=None, generations=200, population_size=300,
                                            tournament_size=30, mutation_rate=0.5, min_mutation=0.4, max_mutation=1.6, zero_prob=0.05,
                                            reflectance=True, early_stopping=5, early_stopping_start=3, early_stopping_tolerance=0.0005,
                                            mandatory_pigments=None, forbidden_pigments=None, expected_recipe=row, uncertanity_bias=0.05,
                                            plot_fitness=False, save_results=True, save_final_result=True, debug=False, plot_best_colors=False, 
                                            save_folder=save_folder, visualization_offset=50, progressbar_text=f'Running genetic algroithm with NN ({additions[c]}) for recipe {i} - {j+1}')
            
            c+=1
    
    with open(f'{save_main_dir}/.done', 'w') as f:
        f.write(f"{datetime.now(ZoneInfo('Europe/Amsterdam')).strftime('%Y-%m-%dT%H:%M:%SZ')}\n")
        f.write('INTERNAL FLAG TO INDICATE THAT THE GA HAS COMPLETED FOR THIS RECIPE!\nDO NOT DELETE OR MODIFY!!!!')






## Evaluation

In [ ]:
from Code.GA.utils import metrics_counter


metrics = metrics_counter(f'{base}/Code/GANN/Results/prc-split-val-small-threshold-04/tuned_model_435/summary.csv')
metrics

## Creation of plots to analyze the performance

In [ ]:
from matplotlib import pyplot as plt


labels = [f'{str(i).zfill(2)}-{str(i+1).zfill(2)}' for i in range(10)]
values = [metrics[f'fitness{i}'] for i in labels]

fig, ax = plt.subplots()
bars = ax.bar(labels, values)

total = metrics['total_values']

# add padding to the y-axis
max_height = max(values)
ax.set_ylim(0, max_height * 1.10)  # increase upper limit

# show values above bars
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2,   # x position
        height,                              # y position
        f'{height}\n{((height/total)*100):.2f}%',                         # text
        ha='center', va='bottom'             # horizontal and vertical alignment
    )

plt.title('Count of fitness values per groups')
plt.xlabel('Group of fitness *10')
plt.ylabel('Count')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


labels = ['0-1', '1-2', '2-4', '4+']
x = np.arange(4)
input1 = [metrics[f'dE94_{i}'] for i in labels]   # values for left y-axis
input2 = [metrics[f'dE94_{i}_pigments_mean'] for i in labels]  # values for right y-axis
input2_err = [metrics[f'dE94_{i}_pigments_std'] for i in labels]  # std dev for red bars

# bar width
width = 0.4

fig, ax1 = plt.subplots()

# add padding to the y-axis
max_height = max(input1)
ax1.set_ylim(0, max_height * 1.10)  # increase upper limit

# plot input1 on ax1
bars1 = ax1.bar(x - width/2, input1, width, label='Input 1', color='tab:blue')
ax1.set_ylabel('Size of each group', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# annotate input1 bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.0f}\n{((height/total)*100):.2f}%',
             ha='center', va='bottom', fontsize=8, color='tab:blue')

# create a second y-axis
ax2 = ax1.twinx()

# plot input2 on ax2 with error bars
bars2 = ax2.bar(
    x + width/2, input2, width,
    yerr=input2_err, capsize=5,
    label='Input 2', color='tab:red',
    error_kw={'linewidth': 0.5, 'capthick': 0.5}  # error bars
)
ax2.set_ylabel('Average amount of pigments in each group', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

# annotate input2 bars with ± std
for bar, err in zip(bars2, input2_err):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 0.01 * height,
             f'{height:.2f}\n±{err:.2f}', ha='center', va='bottom',
             fontsize=8, color='tab:red')

# x-axis settings
plt.xticks(x, labels)
ax1.set_xlabel('dE94 groups')

# title
plt.title('dE94 color distance groups and average pigment count')

# layout
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


labels = ['0-1', '1-2', '2-4', '4+']
x = np.arange(len(labels))

# blue bars: size of each group
input1 = [metrics[f'dE94_{i}'] for i in labels]  # left y-axis

# red bars: mean fitness values
fitness_means = [metrics[f'dE94_{i}_fitness_mean'] for i in labels]  # right y-axis
fitness_stds = [metrics[f'dE94_{i}_fitness_std'] for i in labels]

# bar width
width = 0.4

# figure and axes
fig, ax1 = plt.subplots()

# add padding to the y-axis
max_height = max(input1)
ax1.set_ylim(0, max_height * 1.10)  # increase upper limit

# plot blue bars (group sizes) on ax1
bars1 = ax1.bar(x - width/2, input1, width, color='tab:blue')
ax1.set_ylabel('Size of each group', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# annotate blue bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.0f}\n{((height/total)*100):.2f}%',
             ha='center', va='bottom', fontsize=8, color='tab:blue')

# create second y-axis
ax2 = ax1.twinx()

# plot red bars (fitness means) on ax2 with std as error bars
bars2 = ax2.bar(
    x + width/2, fitness_means, width,
    yerr=fitness_stds, capsize=5,
    color='tab:red',
    error_kw={'linewidth': 0.5, 'capthick': 0.5}
)
ax2.set_ylabel('Mean fitness', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

# annotate red bars with mean ± std
for bar, err in zip(bars2, fitness_stds):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 0.01 * height,
             f'{height:.2f}\n±{err:.2f}', ha='center', va='bottom',
             fontsize=8, color='tab:red')

# x-axis settings
plt.xticks(x, labels)
ax1.set_xlabel('dE94 groups')

# title
plt.title('dE94 color distance groups and mean fitness values')

# layout
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np


labels = ['0-1', '1-2', '2-4', '4+']
x = np.arange(len(labels))

# blue bars: size of each group
input1 = [metrics[f'dE94_{i}'] for i in labels]  # left y-axis

# red bars: mean fitness values
fitness_means = [metrics[f'dE94_{i}_dEm_mean'] for i in labels]  # right y-axis
fitness_stds = [metrics[f'dE94_{i}_dEm_std'] for i in labels]

# bar width
width = 0.4

# figure and axes
fig, ax1 = plt.subplots()

# add padding to the y-axis
max_height = max(input1)
ax1.set_ylim(0, max_height * 1.10)  # increase upper limit

# plot blue bars (group sizes) on ax1
bars1 = ax1.bar(x - width/2, input1, width, color='tab:blue')
ax1.set_ylabel('Size of each group', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')

# annotate blue bars
for bar in bars1:
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.0f}\n{((height/total)*100):.2f}%',
             ha='center', va='bottom', fontsize=8, color='tab:blue')

# create second y-axis
ax2 = ax1.twinx()

# plot red bars (fitness means) on ax2 with std as error bars
bars2 = ax2.bar(
    x + width/2, fitness_means, width,
    yerr=fitness_stds, capsize=5,
    color='tab:red',
    error_kw={'linewidth': 0.5, 'capthick': 0.5}
)
ax2.set_ylabel('Mean dEm', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

# annotate red bars with mean ± std
for bar, err in zip(bars2, fitness_stds):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, height + 0.01 * height,
             f'{height:.2f}\n±{err:.2f}', ha='center', va='bottom',
             fontsize=8, color='tab:red')

# x-axis settings
plt.xticks(x, labels)
ax1.set_xlabel('dE94 groups')

# title
plt.title('dE94 color distance groups and mean dEm values')

# layout
plt.tight_layout()
plt.show()
